# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan006/Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I check two signals on one page in March 2026 before I encode a rule.

The flag-linked signal is CTR versus position (behind FlyRank’s CTR-fix). I bucket pages by March average Search Console position on days where gsc_data_available IS TRUE: 1–3, 4–10, 11–20, 21+. In each bucket I print n and mean CTR from the same month’s measured clicks over measured impressions. Pages with no measured position stay out of this table; I print that leftover n so zeros are not treated as rank 1. I expect mean CTR to fall as the bucket gets worse (1–3 down to 21+). I do not put CTR into the later score: it is the same-window rate as my visibility label.

The second signal is word count versus March impressions. I bucket word_count from dim_content: missing, 1–500, 501–1500, 1501–3000, 3001+. Missing stays its own row (~107k pages had no word count in Week 3). I print n and mean March impressions per bin. I expect longer bins to show higher mean impressions if length travels with visibility; that is a directional check, not a proof.

My draft rule scores how strongly those two safe numbers travel with March visibility so a content strategist can brief “watch these / do not over-read those.” I do not emit a refresh queue.

The rule can output one reason code: assoc_ctrpos_wordcount. The action label is watch_in_brief.

Signal 1 verdict: CONFIRMED. On measured March pages with impressions, mean CTR is highest in positions 1–3 (n = 27,175) and lowest at 21+ (n = 42,851). 4–10 (n = 75,672) and 11–20 (n = 29,606) sit in between and are almost tied, so I do not claim a clean drop at every step — only that worse position buckets do not show higher CTR. That matches the directional CTR-fix story. I still do not put CTR in the later score.

Signal 2 verdict: MIXED. Mean March impressions jump from the 501–1500 bin (n = 69,252, mean ≈ 122) to 1501–3000 (n = 94,594, mean ≈ 1,527), then do not keep rising at 3001+ (n = 60,095, mean ≈ 1,431). The 1–500 bin is unusable (n = 66). Missing word count is common (n = 107,430) and is not “no traffic.” A rule that treats “more words → more visibility” as a smooth score would be wrong. A clearly-explained MIXED here is a win: it saved that version of the rule.

I will encode one rule that can lean on the confirmed position/CTR pattern only as context, using position (not CTR) in the score, and I will not treat word count as a linear booster.

In [1]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

hfToken = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [hfToken],
)

rel = "hf://datasets/FlyRank/internship-warehouse"
factMarch = (
    f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')"
)
dimContent = f"read_parquet('{rel}/dim_content.parquet')"

print("building one row per March page...")

con.sql(f"""
CREATE OR REPLACE TABLE pageMarch AS
SELECT
    f.client_hash_id,
    f.content_hash_id,
    AVG(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END)
        AS avgPosition,
    SUM(f.gsc_impressions) AS marchImpressions,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_clicks ELSE 0 END)
        AS measuredClicks,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_impressions ELSE 0 END)
        AS measuredImpressions,
    MAX(c.word_count) AS wordCount
FROM {factMarch} f
LEFT JOIN {dimContent} c
    ON f.content_hash_id = c.content_hash_id
GROUP BY f.client_hash_id, f.content_hash_id
""")

grain = con.sql("""
SELECT
    COUNT(*) AS pageRows,
    COUNT(DISTINCT content_hash_id) AS distinctPages
FROM pageMarch
""").df()
print("grain check — one page in March 2026")
print(grain.to_string(index=False))

noPosition = con.sql("""
SELECT COUNT(*) AS pagesWithNoMeasuredPosition
FROM pageMarch
WHERE avgPosition IS NULL OR avgPosition <= 0
""").df()
print("pages left out of the CTR table (no measured position)")
print(noPosition.to_string(index=False))

print("signal 1 — CTR vs position (measured days only, impressions > 0)")
ctrBuckets = con.sql("""
SELECT
    CASE
        WHEN avgPosition > 0 AND avgPosition < 4 THEN '1-3'
        WHEN avgPosition >= 4 AND avgPosition < 11 THEN '4-10'
        WHEN avgPosition >= 11 AND avgPosition < 21 THEN '11-20'
        WHEN avgPosition >= 21 THEN '21+'
    END AS positionBucket,
    COUNT(*) AS n,
    SUM(measuredClicks) * 1.0 / SUM(measuredImpressions) AS meanCtr
FROM pageMarch
WHERE avgPosition IS NOT NULL
  AND avgPosition > 0
  AND measuredImpressions > 0
GROUP BY 1
ORDER BY
    CASE positionBucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21+' THEN 4
    END
""").df()
print(ctrBuckets.to_string(index=False))

print("signal 2 — word count vs March impressions")
wordBuckets = con.sql("""
SELECT
    CASE
        WHEN wordCount IS NULL OR wordCount <= 0 THEN 'missing'
        WHEN wordCount <= 500 THEN '1-500'
        WHEN wordCount <= 1500 THEN '501-1500'
        WHEN wordCount <= 3000 THEN '1501-3000'
        ELSE '3001+'
    END AS wordBucket,
    COUNT(*) AS n,
    AVG(marchImpressions) AS meanMarchImpressions
FROM pageMarch
GROUP BY 1
ORDER BY
    CASE wordBucket
        WHEN 'missing' THEN 0
        WHEN '1-500' THEN 1
        WHEN '501-1500' THEN 2
        WHEN '1501-3000' THEN 3
        ELSE 4
    END
""").df()
print(wordBuckets.to_string(index=False))
print("tables done — paste these numbers in chat before writing verdicts")

building one row per March page...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain check — one page in March 2026
 pageRows  distinctPages
   331437         331437
pages left out of the CTR table (no measured position)
 pagesWithNoMeasuredPosition
                      156133
signal 1 — CTR vs position (measured days only, impressions > 0)
positionBucket     n  meanCtr
           1-3 27175 0.003948
          4-10 75672 0.003034
         11-20 29606 0.002961
           21+ 42851 0.001268
signal 2 — word count vs March impressions
wordBucket      n  meanMarchImpressions
   missing 107430            388.831174
     1-500     66            273.939394
  501-1500  69252            121.638119
 1501-3000  94594           1527.221705
     3001+  60095           1430.696114
tables done — paste these numbers in chat before writing verdicts


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I encode one transparent rule on the March page table from section 1. I do not fit weights.

The score is inverted measured position: 1 / avgPosition when Search Console actually measured a position (avgPosition is not null and is greater than 0). Better positions score higher, so a strategist sees first the pages where the confirmed CTR-versus-position pattern is strongest. Pages with no measured position get score 0 and sink to the bottom of the file. That is “not measured,” not rank 1.

I do not put CTR in the score. CTR is the same-window rate as March impressions, and the card forbids label-derived inputs. The CTR table stays context from section 1.

I do not put March impressions in the score. That column is the visibility label.

I do not put word count in the score. Section 1 was MIXED: means jumped at 1501–3000 and did not keep rising at 3001+, and missing length is common and is not “no traffic.” A linear length booster would be wrong. Dropping it is the win from that verdict.

Every row gets one reason code, assoc_position, and one action, watch_in_brief. I retired assoc_ctrpos_wordcount because word count is not in the formula. The action is “include in the strategist brief,” not “refresh this URL.”

I rank every March page, highest score first, and write work/outputs/baseline_action_score.csv from this notebook.

In [2]:
from pathlib import Path
import os

print("scoring every March page by inverted position...")
print("only real ranks count: avgPosition must be at least 1...")
print("leaving CTR, impressions, and word count out of the score...")

con.sql("""
CREATE OR REPLACE TABLE rankedQueue AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY score DESC, content_hash_id, client_hash_id
    ) AS rank,
    client_hash_id,
    content_hash_id,
    score,
    reasonCode,
    action,
    avgPosition
FROM (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN avgPosition IS NOT NULL AND avgPosition >= 1
                THEN 1.0 / avgPosition
            ELSE 0.0
        END AS score,
        'assoc_position' AS reasonCode,
        'watch_in_brief' AS action,
        avgPosition
    FROM pageMarch
)
""")

nRows = con.sql("SELECT COUNT(*) AS n FROM rankedQueue").df()
nScored = con.sql(
    "SELECT COUNT(*) AS n FROM rankedQueue WHERE score > 0"
).df()
print("pages in the queue")
print(nRows.to_string(index=False))
print("pages with a real position score (avgPosition at least 1)")
print(nScored.to_string(index=False))

print("top 5 (hashes only)")
print(
    con.sql("""
        SELECT rank, score, reasonCode, action, avgPosition
        FROM rankedQueue
        WHERE rank <= 5
        ORDER BY rank
    """).df().to_string(index=False)
)

print("current folder is " + os.getcwd())

outDir = Path("work/outputs")
outDir.mkdir(parents=True, exist_ok=True)
outPath = (outDir / "baseline_action_score.csv").resolve()

con.execute(
    f"COPY rankedQueue TO '{outPath.as_posix()}' (HEADER, DELIMITER ',')"
)
print("wrote the ranked queue to " + str(outPath))

scoring every March page by inverted position...
only real ranks count: avgPosition must be at least 1...
leaving CTR, impressions, and word count out of the score...
pages in the queue
     n
331437
pages with a real position score (avgPosition at least 1)
     n
174265
top 5 (hashes only)
 rank  score     reasonCode         action  avgPosition
    1    1.0 assoc_position watch_in_brief          1.0
    2    1.0 assoc_position watch_in_brief          1.0
    3    1.0 assoc_position watch_in_brief          1.0
    4    1.0 assoc_position watch_in_brief          1.0
    5    1.0 assoc_position watch_in_brief          1.0
current folder is /content
wrote the ranked queue to /content/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 all have action = watch_in_brief, reasonCode = assoc_position, avgPosition = 1.0, and score = 1.0. Inverted position did its job: best measured rank first. Confidence is low on every row because measured impressions are 1, 2, or 7 — a single Search Console hit at rank 1, not the CTR-versus-position pattern from section 1 (that table needed a real bucket n). marchImpressions matches measuredImpressions here. What would make the queue wrong as a brief: treating these as “strong visibility associations.” They are the noisiest pages the rule can emit.

Rank 1: watch_in_brief / assoc_position — position 1.0, score 1.0, 2 measured impressions — wrong if those two hits are luck or one query, not a stable rank-1 page.

Rank 2: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if one impression is not enough to mean “best position.”

Rank 3: watch_in_brief / assoc_position — position 1.0, score 1.0, 2 measured impressions — wrong if two hits at rank 1 do not repeat next month.

Rank 4: watch_in_brief / assoc_position — position 1.0, score 1.0, 2 measured impressions — wrong if this is a thin URL, not a brief-worthy association.

Rank 5: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if avgPosition = 1 is a one-row average, not a true top slot.

Rank 6: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if the strategist reads “watch first” as “this page is important.”

Rank 7: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if unmeasured zeros were filtered but a single measured day still dominates the score.

Rank 8: watch_in_brief / assoc_position — position 1.0, score 1.0, 2 measured impressions — wrong if both impressions are bots, internal traffic, or one branded query.

Rank 9: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if rank 1 with n=1 is sampling noise.

Rank 10: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if this page would fall out of the top 20 after a tiny impression floor.

Rank 11: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if “best score” is confused with “most evidence.”

Rank 12: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if the brief treats one hit as the CTR-fix story from section 1.

Rank 13: watch_in_brief / assoc_position — position 1.0, score 1.0, 7 measured impressions — still thin; wrong if 7 hits at rank 1 are still one week of luck, not a pattern.

Rank 14: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if a page with one impression should not outrank every high-traffic page at position 5.

Rank 15: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if hash-tie-breaking is the only reason this row beat rank 16.

Rank 16: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if we claim a unique “why” — it is the same score as ranks 2–12.

Rank 17: watch_in_brief / assoc_position — position 1.0, score 1.0, 2 measured impressions — wrong if two impressions at rank 1 are treated as stronger than rank 13’s seven.

Rank 18: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if missing a volume gate is called a feature, not a hole.

Rank 19: watch_in_brief / assoc_position — position 1.0, score 1.0, 1 measured impression — wrong if this is the page a strategist should open first this week.

Rank 20: watch_in_brief / assoc_position — position 1.0, score 1.0, 7 measured impressions — same hole as rank 13: better than n=1, still not the section 1 bucket story; wrong if 7 impressions count as “confirmed CTR vs position” for this URL.

In [3]:
print("top 20 for the hand review...")
print("impressions are context only — they are not in the score or the csv...")

topTwenty = con.sql("""
SELECT
    q.rank,
    q.score,
    q.reasonCode,
    q.action,
    q.avgPosition,
    p.measuredImpressions,
    p.marchImpressions
FROM rankedQueue q
INNER JOIN pageMarch p
    ON q.client_hash_id = p.client_hash_id
    AND q.content_hash_id = p.content_hash_id
WHERE q.rank <= 20
ORDER BY q.rank
""").df()

print(topTwenty.to_string(index=False))

top 20 for the hand review...
impressions are context only — they are not in the score or the csv...
 rank  score     reasonCode         action  avgPosition  measuredImpressions  marchImpressions
    1    1.0 assoc_position watch_in_brief          1.0                  2.0               2.0
    2    1.0 assoc_position watch_in_brief          1.0                  1.0               1.0
    3    1.0 assoc_position watch_in_brief          1.0                  2.0               2.0
    4    1.0 assoc_position watch_in_brief          1.0                  2.0               2.0
    5    1.0 assoc_position watch_in_brief          1.0                  1.0               1.0
    6    1.0 assoc_position watch_in_brief          1.0                  1.0               1.0
    7    1.0 assoc_position watch_in_brief          1.0                  1.0               1.0
    8    1.0 assoc_position watch_in_brief          1.0                  2.0               2.0
    9    1.0 assoc_position watch_in_brief  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weak pick is not one bad URL. It is the whole top of the queue. Inverted position puts every page with avgPosition = 1 first. The top 20 all have watch_in_brief, assoc_position, score 1.0, and only 1, 2, or 7 measured impressions. That is a single Search Console hit at rank 1, not the section 1 CTR-versus-position story (those buckets needed tens of thousands of pages). I do not change the score. The missing volume gate is the hole.

Three examples:
1. Rank 2: 1 measured impression at position 1.0. Wrong if one hit is treated as “best association.”

2. Rank 14: 1 measured impression at position 1.0. Wrong if this row outranks every high-traffic page at position 5.

3. Rank 13: 7 measured impressions at position 1.0. Better than n=1, still thin. Wrong if 7 hits count as the confirmed CTR-fix pattern for this URL.

No product flags in the score. rankedQueue is inverted avgPosition only (1 / avgPosition when avgPosition >= 1, else 0). I did not copy FlyRank refresh, CTR-fix, or quick-win flag columns into the formula.
No future window. The page table is fact_content_daily_performance for month=2026-03 only. I did not use June or fact_content_daily_performance_sample.
No label-derived inputs. CTR (same-window clicks / impressions) is not in the score. March impressions are the visibility label and are not in the score. Word count is not in the score. Those columns may appear in the top-20 print as context only.

The baseline is frozen for later.

The top 20 all have under 10 measured impressions (20 of 20; min 1, max 7). Ranks 2 and 14 are one hit at position 1. Rank 13 has seven hits and is still not the section 1 bucket story. I did not rewrite the score or the CSV.

The queue columns are rank, hashes, score, assoc_position, watch_in_brief, and avgPosition. CTR, March impressions, word count, and FlyRank product flags are not in the score. The window is March 2026 only.

In [4]:
print("not rewriting the score or the csv...")

thinTop20 = con.sql("""
SELECT
    COUNT(*) AS top20Rows,
    SUM(CASE WHEN p.measuredImpressions < 10 THEN 1 ELSE 0 END)
        AS top20WithUnder10MeasuredImpressions,
    MIN(p.measuredImpressions) AS minMeasuredImpressions,
    MAX(p.measuredImpressions) AS maxMeasuredImpressions
FROM rankedQueue q
INNER JOIN pageMarch p
    ON q.client_hash_id = p.client_hash_id
    AND q.content_hash_id = p.content_hash_id
WHERE q.rank <= 20
""").df()
print("top 20 thin-evidence check")
print(thinTop20.to_string(index=False))

print("three example weak ranks (2, 14, 13)")
print(
    con.sql("""
        SELECT
            q.rank,
            q.score,
            q.reasonCode,
            q.action,
            q.avgPosition,
            p.measuredImpressions
        FROM rankedQueue q
        INNER JOIN pageMarch p
            ON q.client_hash_id = p.client_hash_id
            AND q.content_hash_id = p.content_hash_id
        WHERE q.rank IN (2, 13, 14)
        ORDER BY q.rank
    """).df().to_string(index=False)
)

print("columns stored on the ranked queue (what the csv can contain)")
print(con.sql("DESCRIBE rankedQueue").df().to_string(index=False))

print("leak checklist")
print("score input: inverted avgPosition only (1 / avgPosition when avgPosition >= 1, else 0)")
print("not in the score: CTR, measured clicks, measured impressions, March impressions, word count")
print("not in the score: FlyRank product flag columns")
print("window: month=2026-03 fact table only — not June, not the sample table")

not rewriting the score or the csv...
top 20 thin-evidence check
 top20Rows  top20WithUnder10MeasuredImpressions  minMeasuredImpressions  maxMeasuredImpressions
        20                                 20.0                     1.0                     7.0
three example weak ranks (2, 14, 13)
 rank  score     reasonCode         action  avgPosition  measuredImpressions
    2    1.0 assoc_position watch_in_brief          1.0                  1.0
   13    1.0 assoc_position watch_in_brief          1.0                  7.0
   14    1.0 assoc_position watch_in_brief          1.0                  1.0
columns stored on the ranked queue (what the csv can contain)
    column_name column_type null  key default extra
           rank      BIGINT  YES None    None  None
 client_hash_id     VARCHAR  YES None    None  None
content_hash_id     VARCHAR  YES None    None  None
          score      DOUBLE  YES None    None  None
     reasonCode     VARCHAR  YES None    None  None
         action     VARC

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.